# Part 2 - Real-world Data Fitting

## 1. Hiểu dữ liệu 

### 1.1. Mô tả dữ liệu 

- Bộ dữ liệu gồm 4536 quan trắc, 16 đặc trung mô tả đặc điểm của từng thiên thể
    - pl_orbper - Chu kỳ quỹ đạo
    - pl_orbsmax - Bán trục lớn
    - pl_orbeccen - Độ lệch quỹ đạo
    - pl_trandur - Thời lượng quá cảnh
    - pl_trandep - Độ sâu quá cảnh
    - pl_imppar - 
    - pl_eqt - Nhiệt độ cân bằng
    - pl_insol - Thông lượng bức xạ
    - pl_bmasse - Khối lượng
    - pl_rade - Bán kính hành tinh
    - st_teff - Nhiệt độ sao chủ
    - st_rad - Bán kính sao chủ
    - st_mass - Khối lượng sao chủ
    - st_met - kim loại tính
    - st_logg - log-g
    - sy_dist - hoảng cách tới hệ sao

- Chọn biến mục tiêu đầu ra: pl_rade - Bán kính hành tinh

Động lực vật lý cấu trúc nên một hành tinh khí khổng lồ hoàn toàn khác biệt với một hành tinh đá. Nếu giữ nguyên toàn bộ tập dữ liệu, phương trình mặt phẳng hồi quy sẽ bị nhiễu bởi hiện tượng phương sai không đồng nhất do sự chênh lệch quy mô quá lớn. Nhằm tối ưu hóa khả năng nội suy cục bộ của mô hình, nhóm quyết định lọc và chỉ giữ lại nhóm Hành tinh đá dựa trên ngưỡng phân rã Fulton: Bán kính $\le 1.6 \, R_\oplus$ và Khối lượng $\le 10 \, M_\oplus$. Thao tác này được thực hiện ở tầng thô (raw data), đảm bảo bộ nội suy MICE và các tham số chuẩn hóa (Mean/Std) ở các bước sau chỉ học đúng đặc trưng phân phối của riêng nhóm hành tinh này, từ đó đẩy mức độ chính xác, đồng nhất của tín hiệu lên cao nhất.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'part2':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'part2'))

DATA = ROOT / 'part2' / 'data' / 'planet.csv'
OUT = ROOT / 'part2' / 'output'
DATA, OUT

(WindowsPath('D:/Code/01_University/2526-HK2/TUDTK/Project2_DataFitting/part2/data/planet.csv'),
 WindowsPath('D:/Code/01_University/2526-HK2/TUDTK/Project2_DataFitting/part2/output'))

: 

## 1. EDA va preprocessing

Dataset exoplanet co target lien tuc `pl_rade`. Pipeline thuc hien log transform, winsorization, MICE imputation, standardization, va loc da cong tuyen bang VIF tu `part1.ols_implementation.vif`.

In [3]:
from part2.data_pipeline import run_pipeline

preprocessed = run_pipeline(DATA, OUT, target='pl_rade', make_plots=True)
preprocessed['X_train'].shape, preprocessed['X_test'].shape, preprocessed['feature_names']

[Domain Restriction] Rocky/Super-Earth samples kept: 1084
[DataPipeline] Raw data: rows=867, columns=16, target='pl_rade', missing target rows=0
[DataPipeline] After target cleanup and log-transform: rows=867, features=15, target='log_pl_rade'

[DataPipeline] Log-transform effect on skewness and scale
    source        created  missing  min_before  max_before  skew_before  skew_after
 pl_orbper  log_pl_orbper        0    0.176891   199.66876     6.861484    0.490095
pl_orbsmax log_pl_orbsmax       73    0.005000     0.61930     3.231263    2.594798
pl_trandep log_pl_trandep       40    0.001220     1.65360    10.180175    6.952230
  pl_insol   log_pl_insol       83    0.144000  8385.90000     3.959605   -0.381478
 pl_bmasse  log_pl_bmasse        0    0.036400    10.00000     1.769409   -0.271630
    pl_eqt     log_pl_eqt       67  171.700000  3186.00000     0.878810   -0.476649
   sy_dist    log_sy_dist        6    6.869290  2712.58000     1.182893   -1.150935
   pl_rade    log_pl_rade

((867, 11),
 (217, 11),
 ['pl_orbeccen',
  'pl_trandur',
  'pl_imppar',
  'st_teff',
  'st_rad',
  'st_met',
  'log_pl_orbsmax',
  'log_pl_trandep',
  'log_pl_insol',
  'log_pl_bmasse',
  'log_sy_dist'])

## 2. So sanh mo hinh

Cac mo hinh chinh duoc fit bang code tu `part1`: OLS full, OLS chon bien theo p-value, Ridge chon lambda bang k-fold CV, va Lasso.

In [4]:
from part2.model_comparison import run_model_comparison
model_result = run_model_comparison(OUT / 'preprocessed.pkl', OUT, include_lasso=True)
model_result['summary']

[ModelComparison] Loaded preprocessed data from D:\Code\01_University\2526-HK2\TUDTK\Project2_DataFitting\part2\output\preprocessed.pkl

[ModelComparison] Feature matrix summary
split  rows  features  missing_cells  max_abs_mean  min_std  max_std
train   867        11              0      0.000000 1.000000 1.000000
 test   217        11              0      0.104082 0.929777 1.124674

[ModelComparison] Target distribution
split   n     mean      std      min      q25   median      q75      max
train 867 0.793659 0.124190 0.269874 0.722706 0.819780 0.891998 0.955511
 test 217 0.791408 0.113383 0.451076 0.722706 0.797507 0.883768 0.955511
[ModelComparison] Features used (11): ['pl_orbeccen', 'pl_trandur', 'pl_imppar', 'st_teff', 'st_rad', 'st_met', 'log_pl_orbsmax', 'log_pl_trandep', 'log_pl_insol', 'log_pl_bmasse', 'log_sy_dist']
[ModelComparison] Fitting OLS full: features=11, cv_k=5, params={}
[ModelComparison] OLS full metrics: train_R2=0.6818, test_R2=0.7478, test_RMSE=0.0569, test_MA

[{'model': 'Lasso',
  'test_R2': 0.7478780434609535,
  'test_RMSE': 0.05693163936264646,
  'test_MAE': 0.03794237118315021,
  'cv_MSE': 0.005092613349495762,
  'cv_R2': 0.6727256136358934},
 {'model': 'OLS full',
  'test_R2': 0.747794138424372,
  'test_RMSE': 0.05694111186949687,
  'test_MAE': 0.03795210324407396,
  'cv_MSE': 0.005075361310862147,
  'cv_R2': 0.6734074448343162},
 {'model': 'Ridge',
  'test_R2': 0.7476775598253436,
  'test_RMSE': 0.05695427046163049,
  'test_MAE': 0.03820392514626092,
  'cv_MSE': 0.005073637131173124,
  'cv_R2': 0.6734334846896057},
 {'model': 'OLS selected',
  'test_R2': 0.7439128155396036,
  'test_RMSE': 0.057377586706762365,
  'test_MAE': 0.039012639693033195,
  'cv_MSE': 0.0051030261777122825,
  'cv_R2': 0.6716113862126336}]

## 3. Bonus: Kernel Ridge

Phan bonus dung RBF Kernel Ridge tren mot subset co dinh de tranh ma tran Gram qua lon.

In [5]:
from part2.advanced_methods import run_kernel_ridge_bonus

advanced = run_kernel_ridge_bonus(OUT / 'preprocessed.pkl', OUT, max_train=800)
advanced['best']

Tham số tốt nhất tìm được: {'alpha': 0.001, 'gamma': 0.001}


KeyError: 'best'

## 4. Output files

Cac bang va hinh anh duoc luu trong `part2/output/`: EDA, `model_summary.csv`, `ols_inference.csv`, `model_comparison.png`, `residual_plots.png`, `feature_importance.png`, va cac file JSON tong hop.